# R2AI - Transformers Qwen3-14B LLM Filter

Converted from `scripts/run_openrouter_llm_filter.py` to run locally with Hugging Face `transformers` and `Qwen/Qwen3-14B`.

Pipeline:
1. Load zipped submissions from `old_submission` and vote article candidates per question.
2. Build article text lookups from the SME merged corpus and fallback article corpus.
3. Run Qwen3-14B locally to answer and cite directly relevant articles.
4. Optionally run a second review pass with the same model.
5. Checkpoint JSONL rows, export `results.json`, and create submission zip.

This notebook intentionally has no OpenRouter/API cost estimate cell.


In [ ]:
# Optional install cell for Kaggle/Colab-style environments.
# Run once if the runtime does not already include these packages.
import subprocess
import sys

INSTALL_PACKAGES = False

if INSTALL_PACKAGES:
    pkgs = [
        "transformers>=4.51",
        "accelerate>=0.33",
        "bitsandbytes>=0.43",
        "tqdm",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *pkgs])
    print("Packages installed.")
else:
    print("Skipping package install. Set INSTALL_PACKAGES=True if needed.")


In [ ]:
from __future__ import annotations

import gc
import json
import os
import random
import re
import time
import zipfile
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import torch
from tqdm.auto import tqdm


def _repo_root() -> Path:
    cwd = Path.cwd()
    if (cwd / "old_submission").exists() and (cwd / "data").exists():
        return cwd
    if cwd.name == "notebooks" and (cwd.parent / "old_submission").exists():
        return cwd.parent
    return cwd


REPO_ROOT = _repo_root()
KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
ON_KAGGLE = KAGGLE_INPUT.exists()

# Override these with environment variables when dataset folder names differ.
SUBMISSIONS_DIR = Path(os.getenv(
    "SUBMISSIONS_DIR",
    "/kaggle/input/r2ai-old-submissions" if ON_KAGGLE else str(REPO_ROOT / "old_submission"),
))
SME_MERGED = Path(os.getenv(
    "SME_MERGED",
    "/kaggle/input/r2ai-corpus/corpus_luat_sme_merged_v3.jsonl" if ON_KAGGLE else str(REPO_ROOT / "data/corpus_luat_sme_merged_v3.jsonl"),
))
TH_ARTICLES = Path(os.getenv(
    "TH_ARTICLES",
    "/kaggle/input/r2ai-corpus/th1nhng0_articles_v2.jsonl" if ON_KAGGLE else str(REPO_ROOT / "data/processed/th1nhng0_articles_v2.jsonl"),
))
WORK_DIR = Path(os.getenv("WORK_DIR", str(KAGGLE_WORKING if ON_KAGGLE else REPO_ROOT)))
CHECKPOINT_FILE = Path(os.getenv("CHECKPOINT_FILE", str(WORK_DIR / "data/eval/qwen3_14b_llm_filter_checkpoint.jsonl")))
OUTPUT_JSON = Path(os.getenv("OUTPUT_JSON", str(WORK_DIR / "submissions/results_qwen3_14b_llm_filter.json")))
OUTPUT_ZIP = Path(os.getenv("OUTPUT_ZIP", str(WORK_DIR / "submissions/results_qwen3_14b_llm_filter.zip")))

MODEL_ID = os.getenv("MODEL_ID", "Qwen/Qwen3-14B")
LOAD_IN_4BIT = os.getenv("LOAD_IN_4BIT", "1") == "1"
TRUST_REMOTE_CODE = True

MAX_ARTICLES = int(os.getenv("MAX_ARTICLES", "10"))
MIN_VOTE = int(os.getenv("MIN_VOTE", "1"))
ARTICLE_CHARS = int(os.getenv("ARTICLE_CHARS", "6000"))
MAX_INPUT_TOKENS = int(os.getenv("MAX_INPUT_TOKENS", "24576"))
MAX_NEW_TOKENS = int(os.getenv("MAX_NEW_TOKENS", "1600"))
REVIEW_MAX_NEW_TOKENS = int(os.getenv("REVIEW_MAX_NEW_TOKENS", "1600"))
FALLBACK_TOP = int(os.getenv("FALLBACK_TOP", "3"))
LIMIT = None  # Set to a small integer for smoke tests.
IGNORE_CHECKPOINT = False
REVIEW_PASS = False
THINKING = os.getenv("THINKING", "off")  # "off" is faster and matches enable_thinking=False.
TEMPERATURE = float(os.getenv("TEMPERATURE", "0"))
REVIEW_TEMPERATURE = float(os.getenv("REVIEW_TEMPERATURE", "0"))
RETRIES = int(os.getenv("RETRIES", "1"))
SUBMISSION_KEYS = ("id", "question", "answer", "relevant_docs", "relevant_articles")

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"SUBMISSIONS_DIR: {SUBMISSIONS_DIR}")
print(f"SME_MERGED: {SME_MERGED}")
print(f"TH_ARTICLES: {TH_ARTICLES}")
print(f"MODEL_ID: {MODEL_ID}")
print(f"CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")


In [ ]:
SYSTEM_PROMPT = """\
Bạn là trợ lý pháp lý chuyên về pháp luật Việt Nam dành cho doanh nghiệp vừa và nhỏ.
Nhiệm vụ: trả lời câu hỏi ngắn gọn, chính xác, chỉ dựa trên danh sách điều luật ứng viên.

Quy tắc:
- Chỉ dùng điều luật có căn cứ trực tiếp cho đúng đối tượng và đúng ngữ cảnh trong câu hỏi.
- Không dùng điều luật chỉ liên quan nền, định nghĩa chung, phạm vi áp dụng, thủ tục phụ, hoặc cùng chủ đề nhưng khác đối tượng/ngữ cảnh.
- Nếu nhiều điều luật cùng quy định một hành vi, mức phạt, điều kiện hoặc thời hạn, chỉ dùng văn bản mới hơn/cụ thể hơn; không nhắc văn bản cũ/trùng nếu văn bản mới đã đủ.
- Nếu câu hỏi chỉ hỏi một ý chính, cố gắng dùng 1-2 điều luật. Chỉ dùng 3+ điều luật khi thật sự cần ghép nhiều căn cứ.
- Không trả lời kiểu "Ngoài ra", "đồng thời", "bên cạnh đó" để thêm chính sách khác, trừ khi chính câu hỏi hỏi nhiều nhóm chính sách.
- Nếu câu hỏi hỏi một con số/thời hạn/mức phạt cụ thể, chỉ nêu đúng con số/thời hạn/mức phạt đó và căn cứ trực tiếp.

Bắt buộc trả về đúng định dạng (không được bỏ dòng nào):
CITED: liệt kê tất cả điều luật được dùng, mỗi điều ghi "Điều X ký-hiệu-văn-bản", phân cách bằng dấu chấm phẩy (ví dụ: Điều 9 12/2022/NĐ-CP; Điều 13 04/2017/QH14)
ANSWER: câu trả lời tiếng Việt, tối đa 5 câu
END"""

REVIEW_SYSTEM_PROMPT = """\
Bạn là giám khảo đánh giá câu trả lời pháp lý Việt Nam theo kiểu LLM-as-a-Judge.
Nhiệm vụ: kiểm định và sửa ANSWER dựa duy nhất trên danh sách điều luật ứng viên.

Bạn được cung cấp:
- Câu hỏi gốc.
- Danh sách điều luật ứng viên.
- ANSWER do lượt trước sinh ra.

Tiêu chí đánh giá nội bộ:
1. Correctness: ANSWER có trả lời đúng vấn đề pháp lý được hỏi không?
2. Completeness: ANSWER có đủ các ý chính trong câu hỏi, đặc biệt khi câu hỏi có nhiều vế không?
3. Faithfulness: mọi mệnh đề trong ANSWER có được hỗ trợ trực tiếp bởi điều luật ứng viên không?
4. Citation precision: mỗi điều được nhắc trong ANSWER có trực tiếp cần thiết để trả lời không?
5. Hallucination: ANSWER có thêm điều kiện, mức phạt, thời hạn, thủ tục hoặc tên điều/khoản không có trong điều luật ứng viên không?

Quy tắc chấm và sửa:
- Xoá mọi nội dung không được điều luật ứng viên hỗ trợ trực tiếp.
- Chỉ nhắc điều luật trực tiếp trả lời đúng đối tượng, hành vi, thời hạn, mức phạt, điều kiện hoặc thủ tục trong câu hỏi.
- Loại điều luật chỉ là nền, định nghĩa chung, phạm vi áp dụng, nguyên tắc chung, thủ tục phụ, cùng chủ đề nhưng khác ngữ cảnh.
- Nếu nhiều điều luật trùng cùng hành vi/mức phạt/điều kiện/thời hạn, giữ điều luật mới hơn, cụ thể hơn.
- Nếu câu hỏi có nhiều vế độc lập, giữ đủ điều luật cần thiết cho từng vế.
- Nếu ANSWER dùng "Ngoài ra", "đồng thời", "bên cạnh đó" để thêm chính sách khác không được hỏi, xoá phần đó.
- Có thể bổ sung điều luật ứng viên chưa được nhắc nếu nó trực tiếp cần thiết để ANSWER đủ ý.
- Không dùng kiến thức ngoài danh sách điều luật ứng viên.
- Không xuất giải thích, feedback, điểm số, hay phân tích. Chỉ xuất kết quả cuối cùng.

Bắt buộc trả về đúng định dạng (không được bỏ dòng nào):
CITED: liệt kê tất cả điều luật được dùng, mỗi điều ghi "Điều X ký-hiệu-văn-bản", phân cách bằng dấu chấm phẩy (ví dụ: Điều 9 12/2022/NĐ-CP; Điều 13 04/2017/QH14)
ANSWER: câu trả lời cuối cùng tiếng Việt, tối đa 5 câu
END"""


In [ ]:
@dataclass(frozen=True)
class PreparedQuestion:
    qid: int
    question: str
    articles: list[tuple[str, str]]
    ranked: list[tuple[str, int]]
    messages: list[dict[str, str]]


def apply_thinking_directive(prompt: str, thinking: str) -> str:
    if thinking == "on":
        return f"{prompt}\n/think"
    if thinking == "off":
        return f"{prompt}\n/no_think"
    raise ValueError(f"Unsupported thinking mode: {thinking}")


def article_number(art_str: str) -> str:
    parts = art_str.split("|")
    if len(parts) < 3:
        return ""
    match = re.search(r"\d+", parts[-1])
    return match.group(0) if match else ""


def doc_number(art_str: str) -> str:
    return art_str.split("|", 1)[0]


def clean_submission_doc_str(doc_str: str) -> str:
    doc_str = (doc_str or "").strip()
    parts = doc_str.split("|")
    if len(parts) <= 2:
        return doc_str
    return f"{parts[0]}|{' '.join(parts[1:])}"


def clean_submission_article_str(article_str: str) -> str:
    article_str = (article_str or "").strip()
    parts = article_str.split("|")
    if len(parts) <= 3:
        return article_str
    return f"{parts[0]}|{' '.join(parts[1:-1])}|{parts[-1]}"


def doc_from_article(article_str: str) -> str:
    article_str = clean_submission_article_str(article_str)
    parts = article_str.rsplit("|", 1)
    return clean_submission_doc_str(parts[0] if len(parts) == 2 else article_str)


In [ ]:
def load_all_submissions(submissions_dir: Path) -> dict[int, dict]:
    questions: dict[int, dict] = {}
    zip_files = sorted(submissions_dir.glob("*.zip"))
    print(f"Found {len(zip_files)} submission zips")

    for zpath in zip_files:
        with zipfile.ZipFile(zpath) as z:
            names = [n for n in z.namelist() if n.endswith("results.json")]
            if not names:
                continue
            with z.open(names[0]) as f:
                data = json.load(f)

        for item in data:
            qid = item["id"]
            if qid not in questions:
                questions[qid] = {
                    "question": item["question"],
                    "art_votes": defaultdict(int),
                }
            for art in item.get("relevant_articles", []):
                questions[qid]["art_votes"][clean_submission_article_str(art)] += 1

    print(f"Loaded {len(questions)} unique questions")
    return questions


def build_article_lookups(sme_path: Path, th_path: Path) -> tuple[dict[str, str], dict[tuple[str, str], str]]:
    print("Building primary lookup (corpus_luat_sme_merged_v3) ...")
    primary: dict[str, str] = {}
    with open(sme_path, encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            primary[clean_submission_article_str(r["submission_article_str"])] = r.get("full_text", "") or ""
    print(f"  {len(primary)} articles")

    print("Building fallback lookup (th1nhng0_articles_v2) ...")
    fallback: dict[tuple[str, str], str] = {}
    with open(th_path, encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            fallback[(r["doc_id"], r.get("article_number", ""))] = r.get("full_text", "") or ""
    print(f"  {len(fallback)} articles")
    return primary, fallback


def lookup_text(art_str: str, primary: dict, fallback: dict) -> str:
    art_str = clean_submission_article_str(art_str)
    if art_str in primary:
        return primary[art_str]
    parts = art_str.split("|")
    if len(parts) >= 3:
        doc_id = parts[0]
        raw_article = parts[-1]
        normalized_article = article_number(art_str)
        return (
            fallback.get((doc_id, raw_article), "")
            or fallback.get((doc_id, normalized_article), "")
        )
    return ""


questions = load_all_submissions(SUBMISSIONS_DIR)
primary, fallback = build_article_lookups(SME_MERGED, TH_ARTICLES)


In [ ]:
def load_checkpoint(path: Path) -> dict[int, dict]:
    done: dict[int, dict] = {}
    if not path.exists():
        return done
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            r = json.loads(line)
            r["relevant_articles"] = [clean_submission_article_str(a) for a in r.get("relevant_articles", [])]
            r["relevant_docs"] = [clean_submission_doc_str(d) for d in r.get("relevant_docs", [])]
            done[r["id"]] = r
    print(f"Checkpoint: {len(done)} questions already done")
    return done


def append_checkpoint(path: Path, result: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")


done = {} if IGNORE_CHECKPOINT else load_checkpoint(CHECKPOINT_FILE)
print(f"{len(questions) - len(done)} questions remaining")


In [ ]:
def build_candidates(
    qdata: dict,
    primary: dict,
    fallback: dict,
    max_articles: int,
    min_vote: int,
) -> tuple[list[tuple[str, str]], list[tuple[str, int]]]:
    ranked = sorted(qdata["art_votes"].items(), key=lambda x: -x[1])
    articles: list[tuple[str, str]] = []
    for art_str, votes in ranked:
        if votes < min_vote:
            continue
        if len(articles) >= max_articles:
            break
        text = lookup_text(art_str, primary, fallback)
        if text:
            articles.append((clean_submission_article_str(art_str), text))
    return articles, ranked


def build_messages(
    question: str,
    articles: list[tuple[str, str]],
    article_chars: int,
    thinking: str,
) -> list[dict[str, str]]:
    law_lines = []
    for art_str, text in articles:
        parts = art_str.split("|")
        label = f"{parts[-1]}, {parts[1]}" if len(parts) >= 3 else art_str
        snippet = text[:article_chars].strip()
        law_lines.append(f"{label}\n{snippet}")

    law_block = "\n\n".join(law_lines)
    user_msg = (
        f"ĐIỀU LUẬT ỨNG VIÊN:\n{law_block}\n\n"
        f"CÂU HỎI: {question}\n\n"
        "TRẢ LỜI:"
    )
    return [
        {"role": "system", "content": apply_thinking_directive(SYSTEM_PROMPT, thinking)},
        {"role": "user", "content": user_msg},
    ]


def build_review_messages(
    question: str,
    articles: list[tuple[str, str]],
    article_chars: int,
    selected: list[str],
    answer: str,
    thinking: str,
) -> list[dict[str, str]]:
    law_lines = []
    for art_str, text in articles:
        parts = art_str.split("|")
        label = f"{parts[-1]}, {parts[1]}" if len(parts) >= 3 else art_str
        snippet = text[:article_chars].strip()
        law_lines.append(f"{label}\n{snippet}")

    cited_parts = []
    for art_str in selected:
        parts = art_str.split("|")
        if len(parts) >= 3:
            cited_parts.append(f"{parts[-1]} {parts[0]}")
    cited_line = "; ".join(cited_parts) if cited_parts else "NONE"

    law_block = "\n\n".join(law_lines)
    user_msg = (
        f"CÂU HỎI:\n{question}\n\n"
        f"ĐIỀU LUẬT ỨNG VIÊN:\n{law_block}\n\n"
        f"CÂU TRẢ LỜI CẦN KIỂM ĐỊNH:\n"
        f"CITED: {cited_line}\n"
        f"ANSWER: {answer}\n\n"
        "Hãy kiểm định nghiêm ngặt theo tiêu chí trong system prompt và trả về kết quả cuối cùng:"
    )
    return [
        {"role": "system", "content": apply_thinking_directive(REVIEW_SYSTEM_PROMPT, thinking)},
        {"role": "user", "content": user_msg},
    ]


def prepare_questions(
    questions: dict[int, dict],
    primary: dict,
    fallback: dict,
    done: dict[int, dict],
    *,
    limit: int | None,
    max_articles: int,
    min_vote: int,
    article_chars: int,
    thinking: str,
) -> list[PreparedQuestion]:
    ordered = sorted(questions.items())
    if limit is not None:
        ordered = ordered[:limit]

    prepared = []
    for qid, qdata in ordered:
        if qid in done:
            continue
        articles, ranked = build_candidates(qdata, primary, fallback, max_articles, min_vote)
        messages = build_messages(qdata["question"], articles, article_chars, thinking)
        prepared.append(PreparedQuestion(qid, qdata["question"], articles, ranked, messages))
    return prepared


prepared = prepare_questions(
    questions,
    primary,
    fallback,
    done,
    limit=LIMIT,
    max_articles=MAX_ARTICLES,
    min_vote=MIN_VOTE,
    article_chars=ARTICLE_CHARS,
    thinking=THINKING,
)
print(f"Prepared {len(prepared)} questions")


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    from transformers import BitsAndBytesConfig
except Exception:
    BitsAndBytesConfig = None


def load_qwen3_model(model_id: str):
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=TRUST_REMOTE_CODE)
    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    kwargs: dict[str, Any] = {
        "device_map": "auto",
        "trust_remote_code": TRUST_REMOTE_CODE,
    }
    if torch.cuda.is_available() and LOAD_IN_4BIT and BitsAndBytesConfig is not None:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
    else:
        kwargs["torch_dtype"] = torch.float16 if torch.cuda.is_available() else "auto"

    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    model.eval()
    return model, tokenizer


print(f"Loading {MODEL_ID} with transformers...")
model, tokenizer = load_qwen3_model(MODEL_ID)
print("Model ready.")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: allocated={torch.cuda.memory_allocated(i) / 1e9:.2f} GB")


In [ ]:
def _apply_chat_template(messages: list[dict[str, str]]) -> str:
    kwargs = {
        "tokenize": False,
        "add_generation_prompt": True,
    }
    try:
        return tokenizer.apply_chat_template(
            messages,
            enable_thinking=(THINKING == "on"),
            **kwargs,
        )
    except TypeError:
        return tokenizer.apply_chat_template(messages, **kwargs)


def generate_messages(messages: list[dict[str, str]], max_new_tokens: int, temperature: float) -> str:
    prompt = _apply_chat_template(messages)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    ).to(model.device)

    do_sample = temperature > 0
    generate_kwargs: dict[str, Any] = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if do_sample:
        generate_kwargs["temperature"] = temperature
        generate_kwargs["top_p"] = 0.9

    with torch.inference_mode():
        output = model.generate(**inputs, **generate_kwargs)

    generated_ids = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


def augment_selected_from_answer(
    answer: str,
    articles: list[tuple[str, str]],
    selected: list[str],
) -> list[str]:
    if not answer:
        return selected

    selected_set = set(selected)
    augmented = list(selected)
    answer_norm = answer.lower()
    for art_str, _text in articles:
        if art_str in selected_set:
            continue
        doc = doc_number(art_str)
        no = article_number(art_str)
        if not doc or not no:
            continue
        mentions_doc = doc.lower() in answer_norm
        mentions_article = re.search(rf"Điều\s+{re.escape(no)}\b", answer, re.IGNORECASE)
        if mentions_doc and mentions_article:
            selected_set.add(art_str)
            augmented.append(art_str)
    return augmented


def parse_response(response: str, articles: list[tuple[str, str]]) -> tuple[list[str], str]:
    response = re.sub(r"(?is)<think>.*?</think>", "", response.strip())
    response = re.split(r"(?m)^\s*END\s*$", response, maxsplit=1)[0].strip()

    cited_text = ""
    cited_matches = list(re.finditer(r"(?m)^\s*CITED\s*:\s*([^\n]+)", response))
    if cited_matches:
        cited_text = cited_matches[-1].group(1).strip()

    answer = ""
    answer_matches = list(re.finditer(r"(?m)^\s*ANSWER\s*:\s*([\s\S]+)", response))
    if answer_matches:
        answer = re.split(r"(?m)^\s*CITED\s*:", answer_matches[-1].group(1))[0].strip()

    combined = f"{cited_text}\n{answer}"
    return augment_selected_from_answer(combined, articles, []), answer


def fallback_articles(
    ranked: list[tuple[str, int]],
    primary: dict,
    fallback: dict,
    fallback_top: int,
) -> list[str]:
    return [
        clean_submission_article_str(art_str)
        for art_str, _votes in ranked[:fallback_top]
        if lookup_text(art_str, primary, fallback)
    ]


In [ ]:
def call_with_retries(item: PreparedQuestion) -> tuple[list[str], str, str]:
    last_error = ""
    for attempt in range(RETRIES + 1):
        try:
            raw = generate_messages(item.messages, MAX_NEW_TOKENS, TEMPERATURE)
            selected, answer = parse_response(raw, item.articles)
            return selected, answer, raw
        except Exception as exc:
            last_error = str(exc)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
            if attempt >= RETRIES:
                break
            time.sleep(min(20.0, 1.5 * (2**attempt)) + random.random())
    raise RuntimeError(last_error)


def review_with_retries(item: PreparedQuestion, selected: list[str], answer: str) -> tuple[list[str], str, str]:
    messages = build_review_messages(
        item.question,
        item.articles,
        ARTICLE_CHARS,
        selected,
        answer,
        THINKING,
    )
    last_error = ""
    for attempt in range(RETRIES + 1):
        try:
            raw = generate_messages(messages, REVIEW_MAX_NEW_TOKENS, REVIEW_TEMPERATURE)
            reviewed_selected, reviewed_answer = parse_response(raw, item.articles)
            return reviewed_selected, reviewed_answer, raw
        except Exception as exc:
            last_error = str(exc)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
            if attempt >= RETRIES:
                break
            time.sleep(min(20.0, 1.5 * (2**attempt)) + random.random())
    raise RuntimeError(last_error)


def process_all(prepared: list[PreparedQuestion], done: dict[int, dict]) -> dict[int, dict]:
    errors = 0
    total_llm_s = 0.0
    llm_calls = 0

    pbar = tqdm(prepared, desc="Qwen3-14B QA+cite", unit="q")
    for item in pbar:
        started = time.perf_counter()
        draft_selected: list[str] = []
        draft_answer = ""
        draft_raw = ""
        review_raw = ""

        try:
            selected, answer, raw = call_with_retries(item)
            draft_selected, draft_answer, draft_raw = selected, answer, raw
            if REVIEW_PASS:
                reviewed_selected, reviewed_answer, review_raw = review_with_retries(item, selected, answer)
                if reviewed_answer:
                    selected, answer, raw = reviewed_selected, reviewed_answer, review_raw
            llm_calls += 1
            total_llm_s += time.perf_counter() - started
        except Exception as exc:
            errors += 1
            selected = []
            answer = ""
            raw = ""
            error_text = str(exc)
        else:
            error_text = ""

        if not selected:
            selected = fallback_articles(item.ranked, primary, fallback, FALLBACK_TOP)

        selected = [clean_submission_article_str(a) for a in selected]
        final_docs = list(dict.fromkeys(doc_from_article(a) for a in selected))
        result = {
            "id": item.qid,
            "question": item.question,
            "answer": answer,
            "relevant_docs": final_docs,
            "relevant_articles": selected,
            "_draft_answer": draft_answer,
            "_draft_relevant_articles": draft_selected,
            "_draft_raw": draft_raw,
            "_review_raw": review_raw,
        }
        if error_text:
            result["_error"] = error_text

        done[item.qid] = result
        append_checkpoint(CHECKPOINT_FILE, result)
        avg_s = total_llm_s / llm_calls if llm_calls else 0.0
        pbar.set_postfix(err=errors, cited=len(selected), avg_s=f"{avg_s:.1f}")

    print(f"Done. Errors/fallback rows: {errors}")
    return done


if prepared:
    done = process_all(prepared, done)
else:
    print("Nothing to process.")


In [ ]:
def submission_result(result: dict) -> dict:
    return {key: result.get(key) for key in SUBMISSION_KEYS}


def export(done: dict[int, dict], output_json: Path, output_zip: Path) -> list[dict]:
    results = [submission_result(done[qid]) for qid in sorted(done)]
    output_json.parent.mkdir(parents=True, exist_ok=True)
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"Saved {len(results)} results -> {output_json}")

    output_zip.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(output_json, "results.json")
    print(f"Zipped -> {output_zip}")
    return results


final_results = export(done, OUTPUT_JSON, OUTPUT_ZIP)


In [ ]:
required_keys = {"id", "question", "answer", "relevant_docs", "relevant_articles"}
assert final_results, "No results exported."

for i, row in enumerate(final_results):
    assert required_keys.issubset(row.keys()), f"Missing required keys at row {i}: {row.keys()}"
    assert isinstance(row["id"], int), f"id must be int at row {i}"
    assert isinstance(row["question"], str) and row["question"].strip(), f"empty question at row {i}"
    assert isinstance(row["answer"], str), f"answer must be str at row {i}"
    assert isinstance(row["relevant_docs"], list), f"relevant_docs must be list at row {i}"
    assert isinstance(row["relevant_articles"], list), f"relevant_articles must be list at row {i}"

bad_articles = [
    (r["id"], a)
    for r in final_results
    for a in r["relevant_articles"]
    if not isinstance(a, str) or a.count("|") != 2
]
bad_docs = [
    (r["id"], d)
    for r in final_results
    for d in r["relevant_docs"]
    if not isinstance(d, str) or d.count("|") != 1
]
assert not bad_articles, f"Bad relevant_articles format: {bad_articles[:3]}"
assert not bad_docs, f"Bad relevant_docs format: {bad_docs[:3]}"

empty_articles = [r["id"] for r in final_results if not r["relevant_articles"]]
empty_docs = [r["id"] for r in final_results if not r["relevant_docs"]]
print(f"Rows: {len(final_results)}")
print(f"Empty relevant_articles: {len(empty_articles)}")
print(f"Empty relevant_docs: {len(empty_docs)}")
print("Format relevant_docs/relevant_articles OK.")

sample = final_results[0]
print("\nSample")
print(f"id: {sample['id']}")
print(f"answer[:240]: {sample['answer'][:240]}")
print(f"relevant_docs[:2]: {sample['relevant_docs'][:2]}")
print(f"relevant_articles[:3]: {sample['relevant_articles'][:3]}")
